In [ ]:
!pip install requests==2.32.5 \
             opentelemetry-api==1.40.0 \
             opentelemetry-sdk==1.40.0 \
             opentelemetry-exporter-otlp-proto-common==1.40.0 \
             opentelemetry-proto==1.40.0

In [ ]:
%pip install -qU langchain langchain-community langchain-google-genai chromadb langgraph

In [ ]:
pip install langchain-chroma

In [ ]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

First, mount your Google Drive to access files hosted there.

In [ ]:
from chromadb.config import Settings
from google.colab import userdata
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

import json
import chromadb

GEMINI_API_KEY=userdata.get('GOOGLE_API_KEY')

In [ ]:
import os
from pathlib import Path

if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  file_path = '/content/drive/MyDrive/GenAI/data/dev.json'
  db_path = '/content/drive/MyDrive/GenAI/data/formula_1/formula_1.sqlite'
  vector_db_path = '/content/drive/MyDrive/GenAI/data/chromadb'
else:
  # Define paths for local environment
  # IMPORTANT: Update these paths to match your local setup
  file_path = str(Path('data') / 'dev.json')
  db_path = str(Path('data') / 'formula_1' / 'formula_1.sqlite')
  vector_db_path = str(Path('data') / 'chromadb')

Mounted at /content/drive


Once your Drive is mounted, you can specify the path to your data file. For example, if you have a CSV file named `my_data.csv` in your Drive, you can load it into a pandas DataFrame like this:

In [ ]:
# IMPORTANT: Replace 'path/to/your/my_data.csv' with the actual path to your file
def load_data(file_path):
  try:
    with open(file_path,"r") as f:
      return json.load(f)
  except FileNotFoundError:
    print(f"❌ Error: File not found at {file_path}")
    return None
  except json.JSONDecodeError as e:
    print(f"❌ Error: Could not decode JSON from {file_path}: {e}")
    return None
  except Exception as e:
    print(f"❌ An unexpected error occurred while loading data from {file_path}: {e}")
    return None

In [ ]:
def initialize_gemini_embeddings(api_key: str):
    """Initializes and returns GoogleGenerativeAIEmbeddings."""
    print("✨ Initializing Gemini Embeddings...")
    try:
        return GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=api_key)
    except Exception as e:
        print(f"❌ Error initializing Gemini Embeddings: {e}")
        return None

def get_chroma_client(vector_db_path: str):
    """Initializes and returns a persistent ChromaDB client."""
    print(f"📦 Initializing ChromaDB client at {vector_db_path}...")
    try:
        client = chromadb.PersistentClient(
            path=vector_db_path,
            settings=Settings(anonymized_telemetry=False)
        )
        return client
    except Exception as e:
        print(f"❌ Error initializing ChromaDB client at {vector_db_path}: {e}")
        return None

def get_vector_store(chroma_client, embeddings, collection_name: str = "documents"):
    """
    Initializes and returns a Chroma vector store.
    Deletes existing collection if it exists.
    """
    print(f"🗑️ Deleting existing '{collection_name}' collection (if any)...")
    try:
        chroma_client.delete_collection(collection_name)
        print(f"✅ Collection '{collection_name}' deleted.")
    except Exception as e:
        print(f"⚠️ No existing collection '{collection_name}' found or error deleting: {e}")

    print(f"📚 Initializing Chroma vector store with collection '{collection_name}'...")
    vector_store = Chroma(
        client=chroma_client,
        collection_name=collection_name,
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("✅ Chroma vector store initialized.")
    return vector_store

def prepare_documents_from_dev_data(dev_data: list):
    """
    Prepares a list of Document objects from the raw development data.
    """
    print("📄 Preparing documents from development data...")
    documents = []
    for item in dev_data:
        evidences = [e.strip() for e in item['evidence'].split(';') if e.strip()]
        for evidence in evidences:
            doc = Document(
                page_content=evidence,
                metadata={
                    "db_id": item["db_id"]
                }
            )
            documents.append(doc)
    print(f"✅ Prepared {len(documents)} documents.")
    return documents

def populate_vector_store_with_documents(vector_store, documents):
    """Adds documents to the given vector store."""
    print(f"📚 Loading {len(documents)} text-to-SQL examples into vector store...")
    vector_store.add_documents(documents)
    print(f"✅ Added {len(documents)} text-to-SQL examples to the vector store")

def log_database_distribution(dev_data: list):
    """Calculates and prints the distribution of database IDs in the data."""
    db_counts = {}
    for item in dev_data:
        db_id = item['db_id']
        db_counts[db_id] = db_counts.get(db_id, 0) + 1

    print(f"\n📊 Database distribution:")
    for db_id, count in sorted(db_counts.items()):
        print(f"  - {db_id}: {count} examples")

# Main execution flow
dev_data = load_data(file_path)

if dev_data is None:
    print("❌ Failed to load development data. Exiting.")
else:
    embeddings = initialize_gemini_embeddings(GEMINI_API_KEY)
    if embeddings is None:
        print("❌ Failed to initialize Gemini Embeddings. Exiting.")
    else:
        chroma_client = None # Initialize to None for finally block
        try:
            chroma_client = get_chroma_client(vector_db_path)
            if chroma_client is None:
                print("❌ Failed to initialize ChromaDB client. Exiting.")
            else:
                vector_store = get_vector_store(chroma_client, embeddings)
                # get_vector_store is expected to raise an error if something goes wrong, not return None
                # So, no specific None check here, but the outer try-except will catch issues.

                documents = prepare_documents_from_dev_data(dev_data)
                populate_vector_store_with_documents(vector_store, documents)
                log_database_distribution(dev_data)
        except Exception as e:
            print(f"❌ An unexpected error occurred during database operations: {e}. Exiting.")
        finally:
            if chroma_client:
                print("Closing ChromaDB client...")
                chroma_client.close()
                print("✅ ChromaDB client closed.")